XGBoost - KBest ANOVA

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# Load the dataset
df = pd.read_csv(r'C:\Users\arell\Documents\1_ALF\data\malicious_2021.csv', low_memory=False)

# Select features and target columns
features = ['domain_token_count', 'path_token_count', 'avgdomaintokenlen', 'longdomaintokenlen', 'ldl_domain', 'ldl_path', 
            'subDirLen', 'pathurlRatio', 'argDomanRatio', 'domainUrlRatio', 'NumberofDotsinURL', 'CharacterContinuityRate', 
            'host_DigitCount', 'host_letter_count', 'Directory_LetterCount', 'Domain_LongestWordLength', 
            'sub-Directory_LongestWordLength', 'URLQueries_variable', 'delimeter_Domain', 'delimeter_Count', 
            'NumberRate_Domain', 'SymbolCount_URL', 'SymbolCount_Domain', 'Entropy_Domain', 'tld']

# Clean the dataset by removing NaNs and infinities in numeric columns only
df_cleaned = df.copy()
df_cleaned['tld'] = df_cleaned['tld'].astype(str)
df_cleaned['url_type'] = df_cleaned['url_type'].astype(str)

numeric_features = [f for f in features if f not in ['tld', 'url_type']]
df_cleaned = df_cleaned[np.isfinite(df_cleaned[numeric_features]).all(axis=1)]

# Encode categorical variables
label_encoder_tld = LabelEncoder()
label_encoder_url_type = LabelEncoder()
df_cleaned['tld_encoded'] = label_encoder_tld.fit_transform(df_cleaned['tld'])
df_cleaned['url_type_encoded'] = label_encoder_url_type.fit_transform(df_cleaned['url_type'])
X = df_cleaned[numeric_features + ['tld_encoded']]
y = df_cleaned['binary_label'] = df_cleaned['url_type'].apply(lambda x: 0 if x == 'benign' else 1)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=df_cleaned['binary_label']
)

# Apply K-Best ANOVA to select top 25 features
selector = SelectKBest(score_func=f_classif, k=25)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

# Initialize and fit the XGBoost classifier
xgb_classifier = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_classifier.fit(X_train_selected, y_train)

# Predictions for binary classification
y_train_pred = xgb_classifier.predict(X_train_selected)
y_test_pred = xgb_classifier.predict(X_test_selected)

# Classification reports
print("Binary Classification Report - Training Data:")
print(classification_report(y_train, y_train_pred))
print("Training Accuracy:", accuracy_score(y_train, y_train_pred))

print("\nBinary Classification Report - Test Data:")
print(classification_report(y_test, y_test_pred))
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))

# Multiclass Classification
malicious_df = df_cleaned[df_cleaned['binary_label'] == 1].copy()
X_multi = malicious_df[numeric_features + ['tld_encoded']]

# Encode multiclass labels to start from 0
label_encoder_url_type = LabelEncoder()
y_multi = label_encoder_url_type.fit_transform(malicious_df['url_type'])

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi, test_size=0.4, random_state=42, stratify=y_multi
)

# Select top 25 features for multiclass classification
X_train_multi_selected = selector.fit_transform(X_train_multi, y_train_multi)
X_test_multi_selected = selector.transform(X_test_multi)

# Fit XGBoost for multiclass classification
xgb_multiclass_classifier = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss')
xgb_multiclass_classifier.fit(X_train_multi_selected, y_train_multi)

# Predictions and Evaluations for Multiclass
y_train_pred_multi = xgb_multiclass_classifier.predict(X_train_multi_selected)
y_test_pred_multi = xgb_multiclass_classifier.predict(X_test_multi_selected)

print("Multiclass Classification Report (Training):")
print(classification_report(y_train_multi, y_train_pred_multi))
print("Multiclass Classification Report (Test):")
print(classification_report(y_test_multi, y_test_pred_multi))

# Accuracy Summary
train_accuracy_bin = accuracy_score(y_train, y_train_pred)
test_accuracy_bin = accuracy_score(y_test, y_test_pred)
train_accuracy_multi = accuracy_score(y_train_multi, y_train_pred_multi)
test_accuracy_multi = accuracy_score(y_test_multi, y_test_pred_multi)

print(f"Binary Classification - Train Accuracy: {train_accuracy_bin:.4f}")
print(f"Binary Classification - Test Accuracy: {test_accuracy_bin:.4f}")
print(f"Multiclass Classification - Train Accuracy: {train_accuracy_multi:.4f}")
print(f"Multiclass Classification - Test Accuracy: {test_accuracy_multi:.4f}")


c:\Users\arell\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\xgboost\core.py:158: UserWarning: [16:35:37] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Binary Classification Report - Training Data:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98    299672
           1       0.97      0.94      0.96    156161

    accuracy                           0.97    455833
   macro avg       0.97      0.96      0.97    455833
weighted avg       0.97      0.97      0.97    455833

Training Accuracy: 0.9703575651609252

Binary Classification Report - Test Data:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98    128431
           1       0.97      0.94      0.95     66927

    accuracy                           0.97    195358
   macro avg       0.97      0.96      0.97    195358
weighted avg       0.97      0.97      0.97    195358

Test Accuracy: 0.968836699802414


c:\Users\arell\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\xgboost\core.py:158: UserWarning: [16:38:09] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Multiclass Classification Report (Training):
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     57874
           1       0.99      0.95      0.97     19512
           2       0.98      0.98      0.98     56466

    accuracy                           0.98    133852
   macro avg       0.98      0.97      0.98    133852
weighted avg       0.98      0.98      0.98    133852

Multiclass Classification Report (Test):
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     38583
           1       0.98      0.93      0.96     13008
           2       0.98      0.97      0.97     37645

    accuracy                           0.97     89236
   macro avg       0.98      0.97      0.97     89236
weighted avg       0.97      0.97      0.97     89236

Binary Classification - Train Accuracy: 0.9704
Binary Classification - Test Accuracy: 0.9688
Multiclass Classification - Train Accuracy: 0.9815
Multicla